In [1]:
import ultralytics
from ultralytics import YOLO
import os
import cv2
import time
import torch
import random
import shutil
import tqdm


In [2]:
print("CUDA Available: " + str(torch.cuda.is_available()))
print("Torch CUDA Version: " + str(torch.version.cuda))
# Check to make sure CUDA is available and does not say "None"
ultralytics.utils.checks.collect_system_info()

Ultralytics YOLOv8.1.27 🚀 Python-3.11.8 torch-2.2.1 CUDA:0 (NVIDIA GeForce RTX 3080, 12012MiB)
Setup complete ✅ (24 CPUs, 62.7 GB RAM, 1713.7/1832.2 GB disk)



OS                  Linux-6.8.0-49-generic-x86_64-with-glibc2.35
Environment         Jupyter
Python              3.11.8
Install             pip
RAM                 62.65 GB
CPU                 12th Gen Intel Core(TM) i9-12900KF
CUDA                12.1

matplotlib          ✅ 3.8.3>=3.3.0
opencv-python       ✅ 4.9.0.80>=4.6.0
pillow              ✅ 10.2.0>=7.1.2
pyyaml              ✅ 6.0.1>=5.3.1
requests            ✅ 2.31.0>=2.23.0
scipy               ✅ 1.12.0>=1.4.1
torch               ✅ 2.2.1>=1.8.0
torchvision         ✅ 0.17.1>=0.9.0
tqdm                ✅ 4.66.2>=4.64.0
psutil              ✅ 5.9.8
py-cpuinfo          ✅ 9.0.0
thop                ✅ 0.1.1-2209072238>=0.1.1
pandas              ✅ 2.2.1>=1.1.4
seaborn             ✅ 0.13.2>=0.11.0


In [3]:
CURR_DIR = os.getcwd()
WORKSPACE_DIR = os.path.dirname(CURR_DIR)
DATASETS_DIR = WORKSPACE_DIR + '/datasets/true_negative'
DATA_YAML = WORKSPACE_DIR + '/data.yaml'
CURR_RUN = 'ims_2024_day4_run1_vimba_right'
TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'

In [10]:
def test_boundingbox_model(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """

    bounding_box_results = {}
    size_thresholds = [350,400,450]

    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
    for size_threshold in size_thresholds:
        false_pos = 0
        false_neg = 0
        total_positives = 0
        true_neg = 0
        
        base_save_path = os.path.join(test_results_path, str(size_threshold))

        if not os.path.exists(base_save_path):
            os.makedirs(base_save_path)
        
        # Inference fine-tuned model on test images and save results
        for file in os.listdir(TEST_PATH):
            valid_box_found = False
            file_path = os.path.join(TEST_PATH, file)
            output = model.predict(file_path)
            save_path = os.path.join(base_save_path, file)
            #print(save_path)

            if len(output[0].boxes) == 0:
                true_neg += 1
            else:
                # add bounding box filter threshold
                x1, y1, x2, y2 = output[0].boxes[0].xyxy[0].tolist()
                width = x2-x1
                height = y2-y1
                area = width * height

                if area < size_threshold:
                    true_neg += 1
        
                else:
                    false_pos += 1
                    annotated_img = output[0].plot()
                    save_status = cv2.imwrite(save_path, annotated_img)
                    if not save_status:
                        raise RuntimeError("failed to save")
                
        total_preds = true_neg + false_pos
        if total_preds > 0:
            precision = true_neg / (total_preds)
        # Log results with folder path
        log_message = (
            f'\nTest Folder: {TEST_PATH}\n'
            f'Bounding box size threshold: {size_threshold}, True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision:.4f}\n'
        )
        with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
            log_file.write(log_message)
        
        print(log_message.strip())  # Print to console as well

        bounding_box_results[size_threshold] = [true_neg, false_pos, precision]

    return bounding_box_results

In [5]:
def test_model(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """
    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    false_pos = 0
    true_neg = 0

    # Inference fine-tuned model on test images and save results
    for file in os.listdir(TEST_PATH):
        file_path = os.path.join(TEST_PATH, file)
        output = model.predict(file_path)
        save_path = os.path.join(test_results_path, file)
        print(output[0].boxes)
        if len(output[0].boxes) == 0:
            true_neg += 1
        else:
            # Filter boxes by confidence > 50%
            high_conf_boxes = [box for box in output[0].boxes if box.conf > 0.5]
            if len(high_conf_boxes) > 0:
                false_pos += 1

                # Save annotated image with bounding boxes
                annotated_img = output[0].plot()
                cv2.imwrite(save_path, annotated_img)

    total_preds = true_neg + false_pos
    if total_preds > 0:
        precision = true_neg / (total_preds)
    # Log results with folder path
    log_message = (
        f'\nTest Folder: {TEST_PATH}\n'
        f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision:.4f}\n'
    )
    with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
        log_file.write(log_message)
    
    print(log_message.strip())  # Print to console as well
    return true_neg, false_pos, precision

In [11]:
#initialize model
runs = ['ims_2024_day6_run2_front']

for run in runs:
    CURR_RUN = run

    CURR_MODEL_PATH = WORKSPACE_DIR + '/models/yolo_model_files/epoch95.pt'
    model = YOLO(CURR_MODEL_PATH)
    data_dir = CURR_DIR + '/../data/'

    # initialize test results
    test_results_path = CURR_DIR + f'/../results/output_imgs/{CURR_RUN}'
    bb_results = test_boundingbox_model(model, test_results_path)



image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day6_run2_front/images/Image_0000018001_sec1725471171_nsec54198274.jpg: 800x1056 (no detections), 5.1ms
Speed: 3.3ms preprocess, 5.1ms inference, 0.3ms postprocess per image at shape (1, 3, 800, 1056)

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day6_run2_front/images/Image_0000016761_sec1725471109_nsec53521784.jpg: 800x1056 (no detections), 5.2ms
Speed: 2.8ms preprocess, 5.2ms inference, 0.4ms postprocess per image at shape (1, 3, 800, 1056)

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day6_run2_front/images/Image_0000014546_sec1725470998_nsec301750899.jpg: 800x1056 (no detections), 3.9ms
Speed: 2.3ms preprocess, 3.9ms inference, 0.2ms postprocess per image at shape (1, 3, 800, 1056)

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day6_run2_front/images/Image_0000025886_sec1725471607_nsec82

In [ ]:
#initialize model
CURR_MODEL_PATH = WORKSPACE_DIR + '/models/yolo_model_files/epoch95.pt'
model = YOLO(CURR_MODEL_PATH)
data_dir = CURR_DIR + '/../data/'

# initialize test results
test_results_path = CURR_DIR + f'/../results/annotated_imgs/{CURR_RUN}'
true_neg, false_pos, precision = test_model(model, test_results_path)



image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day4_run1_vimba_right/images/frame_002783.PNG: 800x1056 (no detections), 3.8ms
Speed: 2.4ms preprocess, 3.8ms inference, 0.2ms postprocess per image at shape (1, 3, 800, 1056)
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([], device='cuda:0')
conf: tensor([], device='cuda:0')
data: tensor([], device='cuda:0', size=(0, 6))
id: None
is_track: False
orig_shape: (772, 1032)
shape: torch.Size([0, 6])
xywh: tensor([], device='cuda:0', size=(0, 4))
xywhn: tensor([], device='cuda:0', size=(0, 4))
xyxy: tensor([], device='cuda:0', size=(0, 4))
xyxyn: tensor([], device='cuda:0', size=(0, 4))

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day4_run1_vimba_right/images/frame_003408.PNG: 800x1056 (no detections), 3.4ms
Speed: 2.1ms preprocess, 3.4ms inference, 0.2ms postprocess per image at shape (1, 3, 800, 1056)
ultralytics.engine.results.Boxes object wit

KeyboardInterrupt: 

In [68]:
# ims day 5
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 7186, False_Pos: 134, Precision: 0.9816939890710382


In [17]:
# ims day 6 run 1 rear
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 808, False_Pos: 1, Precision: 0.9987639060568603


In [71]:
# ims day 6 run 2
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 2975, False_Pos: 93, Precision: 0.9696870925684485


In [ ]:
# ims day 6 run 1: 808 no detections, 1 car detection, 99.87% precise
# ims day 6 run 2: 2975 no detections, 98 car detections, 95.97% precise 
# ims day 5: 7186 no detections, 134 car detections, 98.17% precise